# Week 5 — Learning a Pricing Function

## Overview

**Research question:** With the same simulated reference contracts, does a smoother supervised predictor improve valuation?

We hold the compressor and its MC labels fixed. We compare nearest-reference copying and a three-neighbor average with Ridge regression, an additive spline model, and a random forest. Gradient boosting is an optional extension. The goal is to understand the assumptions each model makes and to compare them without changing the available information.

**Learning objectives**

- Express pricing as supervised regression with noisy simulation labels.
- Explain the difference between linear, additive nonlinear, and tree-based predictors.
- Tune a small declared model grid using validation MC labels.
- Separate training error, validation error, and final test error.
- Report preprocessing, label, fitting, tuning, and prediction costs.

**Prerequisites:** Weeks 1–4; linear combinations, squared error, and the compressor–simulator–predictor framework. No prior machine learning course is required.

**Output contract:** a development leaderboard, a table of tuning trials, prediction plots, a cost ledger, and a justified choice of models to carry into the controlled benchmark.

**Suggested workload:** 75-minute meeting plus 4–6 independent hours. Complete the three core model families; boosting is optional.

**Study boundary:** all labels are normalized gross GMAB top-up liabilities at a specified fee. Direct prediction models do not automatically learn a contract-distance metric. The test split remains closed until Week 8.

## 1. What is being learned?

The reference set is \(\{(z_i,\widehat y_i)\}_{i=1}^{m}\), where \(z_i\) contains five candidate-scaled contract features and \(\widehat y_i\) is a Monte Carlo mean. A supervised predictor fits a function \(\widehat f(z)\) and returns

\[
\widehat y(q)=\widehat f(z(q)),\qquad
\widehat L(q)=P_q\,\widehat f(z(q)).
\]

A common fitting principle is to minimize squared discrepancies plus a complexity penalty:

\[
\sum_{i=1}^{m}\left[\widehat y_i-f(z_i)\right]^2
+\text{penalty}(f).
\]

The labels satisfy \(\widehat y_i=y_i+\varepsilon_i\). Their noise variances differ across contracts. For this introductory comparison we use unweighted fitting and report MC standard errors; changing label weighting would be another experimental factor.

There are two distinct research ideas:

| Idea | What is fitted? | Example |
|---|---|---|
| Direct value prediction | A mapping from features to liability | Ridge, additive splines, random forest |
| Response-aware similarity | A distance that helps choose comparable contracts | Week 6's learned diagonal distance |

A good direct predictor may reveal useful response patterns, but its fitted coefficients or tree rules are not automatically a valid metric. This week asks which prediction method works with a fixed label bank.

The target remains
\(y=E_Q[e^{-rT}(g-A_T/P)_+]\).
It excludes mortality, lapses, withdrawals, ratchets, and fee-income deductions. Do not interpret a prediction as a full variable annuity price or a fair fee.

In [ ]:
# Complete common pricing code. No data files or earlier notebooks are required.
import sys
import json
import platform
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

try:
    from IPython.display import display
except ImportError:
    def display(*objects):
        for obj in objects:
            print(obj)

RISK_FREE_RATE = 0.03
FEATURES = ['account_ratio', 'guarantee_ratio', 'maturity', 'volatility', 'fee']
plt.rcParams.update({'figure.figsize': (7.2, 4.3), 'axes.grid': True,
                     'grid.alpha': 0.22, 'font.size': 10})


def make_portfolio(n=1200, seed=2027):
    """Synthetic in-force GMAB contracts, never customer data.

    P is a reference premium, A0=P*account_ratio, and G=P*guarantee_ratio.
    T is the remaining term. All contracts share the same risk-free rate.
    Volatility describes the contract's fund; rates are annual decimals.
    """
    if not isinstance(n, (int, np.integer)) or n < 10:
        raise ValueError('n must be an integer of at least 10.')
    rng = np.random.default_rng(seed)
    return pd.DataFrame({
        'contract_id': np.arange(n, dtype=int),
        'premium': rng.uniform(10_000.0, 50_000.0, n),
        'account_ratio': rng.uniform(0.65, 1.40, n),
        'guarantee_ratio': rng.uniform(0.90, 1.60, n),
        'maturity': rng.integers(5, 21, n).astype(float),
        'volatility': rng.uniform(0.12, 0.35, n),
        'fee': rng.uniform(0.005, 0.030, n),
    })


def validate_contracts(df):
    required = ['contract_id', 'premium'] + FEATURES
    missing = [c for c in required if c not in df]
    if missing:
        raise ValueError(f'Missing contract columns: {missing}')
    if len(df) == 0:
        raise ValueError('Supply at least one contract.')
    values = df[required].to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError('All contract fields must be finite.')
    ids = df['contract_id'].to_numpy(dtype=float)
    if (ids < 0).any() or (ids != np.floor(ids)).any() or len(np.unique(ids)) != len(ids):
        raise ValueError('contract_id must contain unique nonnegative integers.')
    if (df[['premium', 'account_ratio', 'guarantee_ratio']] <= 0).any().any():
        raise ValueError('Premium and account/guarantee ratios must be positive.')
    if (df[['maturity', 'volatility', 'fee']] < 0).any().any():
        raise ValueError('Maturity, volatility and fee must be nonnegative.')


def gmab_mc(df, n_paths=2048, seed=123):
    """Independent plain Monte Carlo for per-premium gross top-up liability.

    Identical (seed, contract_id) reproduces the same path stream even when
    the contract appears in a different subset or row order. Distinct IDs
    give independent streams. This is NOT a common-market portfolio-risk
    simulator: it estimates a sum of expected liabilities, not portfolio VaR.
    SE describes Monte Carlo sampling uncertainty only.
    """
    validate_contracts(df)
    if not isinstance(n_paths, (int, np.integer)) or n_paths < 2:
        raise ValueError('n_paths must be an integer of at least 2.')
    if not isinstance(seed, (int, np.integer)) or seed < 0:
        raise ValueError('seed must be a nonnegative integer.')
    values, errors, sds = [], [], []
    for row in df.itertuples():
        rng = np.random.default_rng(np.random.SeedSequence([int(seed), int(row.contract_id)]))
        z = rng.standard_normal(n_paths)
        terminal_ratio = row.account_ratio * np.exp(
            (RISK_FREE_RATE - row.fee - 0.5 * row.volatility**2) * row.maturity
            + row.volatility * np.sqrt(row.maturity) * z
        )
        payoff = np.exp(-RISK_FREE_RATE * row.maturity) * np.maximum(
            row.guarantee_ratio - terminal_ratio, 0.0)
        sd = float(np.std(payoff, ddof=1))
        values.append(float(np.mean(payoff)))
        errors.append(sd / np.sqrt(n_paths))
        sds.append(sd)
    return pd.DataFrame({'value': values, 'se': errors, 'payoff_sd': sds,
                         'paths': int(n_paths)}, index=df.index)


def gmab_exact(df):
    """Closed-form put benchmark under the simplified model: audit only.

    Never use these values as training or tuning labels in the project.
    The deterministic cases T=0 or sigma=0 are handled separately.
    """
    validate_contracts(df)
    a = df['account_ratio'].to_numpy(dtype=float)
    g = df['guarantee_ratio'].to_numpy(dtype=float)
    t = df['maturity'].to_numpy(dtype=float)
    sig = df['volatility'].to_numpy(dtype=float)
    fee = df['fee'].to_numpy(dtype=float)
    ans = np.maximum(g * np.exp(-RISK_FREE_RATE*t) - a * np.exp(-fee*t), 0.0)
    regular = (t > 0) & (sig > 0)
    if np.any(regular):
        tr, sr = t[regular], sig[regular]
        d1 = (np.log(a[regular] / g[regular]) +
              (RISK_FREE_RATE-fee[regular]+0.5*sr**2)*tr) / (sr*np.sqrt(tr))
        d2 = d1 - sr*np.sqrt(tr)
        ans[regular] = (g[regular]*np.exp(-RISK_FREE_RATE*tr)*norm.cdf(-d2)
                        - a[regular]*np.exp(-fee[regular]*tr)*norm.cdf(-d1))
    return np.maximum(ans, 0.0)


def split_portfolio(df, seed=2027):
    """Split contracts before building pairs, selecting labels or tuning."""
    validate_contracts(df)
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(df))
    n_candidate, n_validation = int(0.60*len(df)), int(0.20*len(df))
    if min(n_candidate, n_validation, len(df)-n_candidate-n_validation) < 1:
        raise ValueError('Portfolio is too small for the requested split.')
    return {
        'candidate': df.iloc[order[:n_candidate]].copy(),
        'validation': df.iloc[order[n_candidate:n_candidate+n_validation]].copy(),
        'test': df.iloc[order[n_candidate+n_validation:]].copy(),
    }


def score_predictions(y_true, y_pred, premium=None):
    """Errors per premium, plus signed aggregate relative error when requested."""
    truth = np.asarray(y_true, dtype=float).reshape(-1)
    pred = np.asarray(y_pred, dtype=float).reshape(-1)
    if truth.shape != pred.shape or len(truth) == 0:
        raise ValueError('Truth and predictions must have equal nonzero length.')
    if not (np.isfinite(truth).all() and np.isfinite(pred).all()):
        raise ValueError('Truth and predictions must be finite.')
    error = pred-truth
    out = {'rmse': float(np.sqrt(np.mean(error**2))),
           'mae': float(np.mean(np.abs(error))),
           'p95_abs': float(np.quantile(np.abs(error), 0.95)),
           'bias': float(np.mean(error)),
           'negative_fraction': float(np.mean(pred < 0))}
    if premium is not None:
        p = np.asarray(premium, dtype=float).reshape(-1)
        if p.shape != truth.shape or not np.isfinite(p).all() or np.any(p <= 0):
            raise ValueError('Premium weights must be finite, positive and aligned.')
        denominator = float(p @ truth)
        out['portfolio_relative_error'] = (float(p @ error)/denominator
                                            if abs(denominator) > 1e-12 else np.nan)
    return out


In [ ]:
# Complete shared learning code. All labels are supplied explicitly by the caller.
from scipy.spatial.distance import cdist
from scipy.optimize import minimize
from sklearn.preprocessing import MinMaxScaler, SplineTransformer
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import make_pipeline


def build_features(candidate, query=None, columns=FEATURES):
    """Fit feature scales only on the unlabelled candidate pool.

    A constant training column maps to zero on training data. A held-out
    observation outside the training range may map outside [0,1]; no clipping.
    No premium, contract ID or computed valuation is an informative feature.
    """
    scaler = MinMaxScaler()
    z_candidate = scaler.fit_transform(candidate[list(columns)])
    z_query = None if query is None else scaler.transform(query[list(columns)])
    return scaler, z_candidate, z_query


def select_representatives(Z, m, method='kmeans', seed=2027):
    """Return m DISTINCT row positions of actual candidate contracts.

    K-means is fitted to candidate FEATURES only. For each nonempty cluster,
    choose the member closest to that cluster's center. This is a member
    representative, not an exact k-medoids optimization or Feng et al.'s SRSC.
    """
    z = np.asarray(Z, dtype=float)
    if z.ndim != 2 or not np.isfinite(z).all():
        raise ValueError('Z must be a finite 2D matrix.')
    if not isinstance(m, (int, np.integer)) or not 1 <= m <= len(z):
        raise ValueError('m must lie between 1 and the number of candidates.')
    rng = np.random.default_rng(seed)
    if method == 'random':
        return np.sort(rng.choice(len(z), size=m, replace=False))
    if method != 'kmeans':
        raise ValueError("method must be 'random' or 'kmeans'.")
    km = KMeans(n_clusters=m, n_init=5, max_iter=150, random_state=seed)
    membership = km.fit_predict(z)
    chosen = []
    for cluster in range(m):
        members = np.flatnonzero(membership == cluster)
        if len(members):
            distances = np.sum((z[members] - km.cluster_centers_[cluster])**2, axis=1)
            chosen.append(int(members[np.argmin(distances)]))
    # Duplicate feature vectors can yield fewer than m nonempty clusters.
    # Keep the requested contract count without ever repeating an ID.
    if len(chosen) < m:
        remaining = np.setdiff1d(np.arange(len(z)), chosen)
        chosen.extend(rng.choice(remaining, m-len(chosen), replace=False).tolist())
    return np.sort(np.asarray(chosen, dtype=int))


def nearest_predict(Zref, yref, Zquery, weights=None, k=1):
    """Uniform k-neighbor transfer; k=1 is the CSM-style teaching baseline.

    A diagonal weighted distance is ordinary Euclidean distance after each
    coordinate is multiplied by sqrt(weight). Stable sort breaks exact ties
    in reference-row order. Chunking avoids an N-by-N distance matrix.
    """
    zr, zq = np.asarray(Zref, dtype=float), np.asarray(Zquery, dtype=float)
    yr = np.asarray(yref, dtype=float).reshape(-1)
    if zr.ndim != 2 or zq.ndim != 2 or zr.shape[1] != zq.shape[1] or len(zr) != len(yr):
        raise ValueError('Reference/query dimensions or labels do not align.')
    if not (np.isfinite(zr).all() and np.isfinite(zq).all() and np.isfinite(yr).all()):
        raise ValueError('Features and labels must be finite.')
    if not isinstance(k, (int, np.integer)) or not 1 <= k <= len(zr):
        raise ValueError('k must be between 1 and the reference count.')
    w = np.ones(zr.shape[1]) if weights is None else np.asarray(weights, dtype=float)
    if w.shape != (zr.shape[1],) or not np.isfinite(w).all() or np.any(w < 0):
        raise ValueError('Distance weights must be finite and nonnegative.')
    scale = np.sqrt(w)
    neighbors = np.empty((len(zq), k), dtype=int)
    for start in range(0, len(zq), 1024):
        stop = min(start+1024, len(zq))
        distance = cdist(zq[start:stop]*scale, zr*scale, metric='sqeuclidean')
        neighbors[start:stop] = np.argsort(distance, axis=1, kind='stable')[:, :k]
    return yr[neighbors].mean(axis=1), neighbors


def fit_metric(Zref, yref, lam=0.01, max_pairs=2000, seed=2027):
    """Fit a simple diagonal response-aware pseudometric on reference pairs.

    Objective: mean(((H@w)/p - ((y_i-y_j)/s)^2)^2)
               + lam/2 * sum((w-1)^2),  w>=0, sum(w)=p.
    Here H=(z_i-z_j)^2 coordinatewise and s=max(ptp(yref),1e-6).
    The label scale is part of the modeling choice, fitted on references only.
    Numerical labels have noise: this loss is not an unbiased exact-gap loss.
    """
    z, y = np.asarray(Zref, dtype=float), np.asarray(yref, dtype=float).reshape(-1)
    if z.ndim != 2 or len(z) != len(y) or len(y) < 2:
        raise ValueError('Supply at least two aligned labeled reference contracts.')
    if not (np.isfinite(z).all() and np.isfinite(y).all()):
        raise ValueError('Training data must be finite.')
    if lam < 0 or not isinstance(max_pairs, (int, np.integer)) or max_pairs < 1:
        raise ValueError('lam must be nonnegative and max_pairs a positive integer.')
    n, p = z.shape
    count = min(max_pairs, n*(n-1)//2)
    rng = np.random.default_rng(seed)
    if count == n*(n-1)//2:
        i, j = np.triu_indices(n, k=1)
    else:
        # Uniform directed non-self pairs sampled WITH replacement.
        # They share contracts and are not independent observational units.
        i = rng.integers(0, n, size=count)
        j = (i+rng.integers(1, n, size=count)) % n
    label_scale = max(float(np.ptp(y)), 1e-6)
    h = (z[i]-z[j])**2 / p
    target = ((y[i]-y[j])/label_scale)**2
    def objective(w):
        residual = h @ w-target
        return float(np.mean(residual**2) + 0.5*lam*np.sum((w-1.0)**2))
    def gradient(w):
        return 2.0*h.T @ (h @ w-target)/len(target) + lam*(w-1.0)
    result = minimize(objective, np.ones(p), jac=gradient, method='SLSQP',
                      bounds=[(0.0, float(p))]*p,
                      constraints=[{'type': 'eq', 'fun': lambda w: w.sum()-p,
                                    'jac': lambda w: np.ones(p)}],
                      options={'maxiter': 400, 'ftol': 1e-10})
    feasible = (np.isfinite(result.x).all() and result.x.min() >= -1e-7
                and abs(result.x.sum()-p) <= 1e-6)
    success = bool(result.success and feasible)
    weights = np.clip(result.x, 0.0, None) if success else np.ones(p)
    weights *= p/weights.sum()
    return {'weights': weights, 'success': success, 'pair_count': len(target),
            'objective': objective(weights), 'scale': label_scale,
            'iterations': int(result.nit), 'message': str(result.message)}


def make_regressors(seed=2027):
    """Fixed small model menu. Regression predicts values, not distances."""
    return {
        'Ridge': Ridge(alpha=1.0),
        'SplineGAM': make_pipeline(
            SplineTransformer(n_knots=4, degree=3, include_bias=False),
            Ridge(alpha=1.0)),
        'RandomForest': RandomForestRegressor(n_estimators=100, min_samples_leaf=3,
                                               random_state=seed, n_jobs=1),
        'GradientBoosting': GradientBoostingRegressor(n_estimators=100, max_depth=2,
                                                       learning_rate=0.05, random_state=seed),
    }


In [ ]:
from sklearn.base import clone

PORTFOLIO_SEED = 2027
SPLIT_SEED = 2027
COMPRESSOR_SEED = 2027
REFERENCE_MC_SEED = 4101
VALIDATION_MC_SEED = 4102
N_PORTFOLIO = 1000
N_REFERENCES = 40
N_VALIDATION = 100
REFERENCE_PATHS = 2048
VALIDATION_PATHS = 2048
INCLUDE_OPTIONAL_BOOSTING = False

portfolio = make_portfolio(n=N_PORTFOLIO, seed=PORTFOLIO_SEED)
parts = split_portfolio(portfolio, seed=SPLIT_SEED)
candidate = parts["candidate"].copy()
validation = parts["validation"].sample(
    n=min(N_VALIDATION, len(parts["validation"])), random_state=2028
).copy()
# Keep parts["test"] closed. No exact-price function is called.

t0 = perf_counter()
scaler, Zcandidate, Zvalidation = build_features(candidate, validation)
feature_seconds = perf_counter() - t0

t0 = perf_counter()
reference_positions = select_representatives(
    Zcandidate, m=N_REFERENCES, method="kmeans", seed=COMPRESSOR_SEED
)
compression_seconds = perf_counter() - t0
references = candidate.iloc[reference_positions].copy()
Zreference = Zcandidate[reference_positions]

t0 = perf_counter()
reference_mc = gmab_mc(
    references, n_paths=REFERENCE_PATHS, seed=REFERENCE_MC_SEED
)
reference_mc_seconds = perf_counter() - t0
y_reference = reference_mc["value"].to_numpy()

t0 = perf_counter()
validation_mc = gmab_mc(
    validation, n_paths=VALIDATION_PATHS, seed=VALIDATION_MC_SEED
)
validation_mc_seconds = perf_counter() - t0
y_validation = validation_mc["value"].to_numpy()
premium_validation = validation["premium"].to_numpy()
upper_bound_validation = (
    validation["guarantee_ratio"].to_numpy()
    * np.exp(-RISK_FREE_RATE * validation["maturity"].to_numpy())
)

label_preview = references[["contract_id"] + FEATURES].copy()
label_preview["MC_liability_per_premium"] = y_reference
label_preview["MC_SE_per_premium"] = reference_mc["se"].to_numpy()
display(label_preview.head(5))
print("Shared reference labels:", len(y_reference))
print("Shared validation labels:", len(y_validation))
print("All predictors below use these same rows and these same numeric labels.")

## 2. Ridge regression: start with a linear response

A linear predictor has the form

\[
f(z)=\beta_0+\sum_{j=1}^{p}\beta_j z_j.
\]

Ridge regression estimates the coefficients by minimizing

\[
\sum_{i=1}^{m}
\left(\widehat y_i-\beta_0-z_i^\top\beta\right)^2
+\alpha\sum_{j=1}^{p}\beta_j^2.
\]

The intercept is not penalized in our implementation. A larger \(\alpha\) shrinks the slopes more strongly. The penalty can stabilize fitting with a small reference sample, at the cost of bias. This is the sum-of-squares convention used by the estimator; using a mean loss would require rescaling \(\alpha\).

**Worked example.** The illustrative function
\(f(z_1,z_2)=0.01+0.02z_1+0.06z_2\)
predicts \(0.056\) at \((0.2,0.7)\): a normalized liability of 5.6% of premium. These are invented coefficients for arithmetic, not fitted results.

A linear response has the same slope at every input. Our payoff model contains interactions such as \(\phi T\) and \(\sigma\sqrt T\), so a simple linear approximation can miss structure. Nevertheless, an inexpensive, stable linear baseline is a useful comparator.

A coefficient reflects this fitted model and this feature scale. It is not a causal effect, a global sensitivity theorem, or automatically a distance weight. A linear model can also produce negative liabilities; we will count such predictions rather than silently changing the rule after seeing results.

In [ ]:
illustrative_z = np.array([0.2, 0.7])
illustrative_beta = np.array([0.02, 0.06])
illustrative_prediction = 0.01 + illustrative_z @ illustrative_beta
print("Worked-example prediction per premium:", illustrative_prediction)
print("Worked-example currency prediction at P=20,000:",
      20_000 * illustrative_prediction)

## 3. An additive spline model: nonlinear, but still structured

An additive model replaces each linear feature contribution with a smooth function:

\[
f(z)=\beta_0+\sum_{j=1}^{p}f_j(z_j),
\qquad
f_j(z_j)=\sum_{\ell=1}^{K_j}\theta_{j\ell}B_{j\ell}(z_j).
\]

The \(B_{j\ell}\) are spline basis functions. They act as a flexible set of building blocks for a smooth curve. After creating these columns, fitting the coefficients is another regularized linear regression problem.

Our **SplineGAM** is a Gaussian identity-link additive regression model: the conditional mean is modelled directly as a sum of smooth contributions, and squared error is the fitting loss. This name describes its mean model and loss; it does not assert that finite-sample MC label noise is exactly Gaussian. The code uses cubic splines and a Ridge penalty on coefficients. It is not a full GAM package with automatic smoothness selection or inference.

The model is nonlinear in the original features, but it has **no learned cross-feature interactions**. For example, an additive \(f_1(\phi)+f_2(T)\) cannot generally reproduce a term proportional to \(\phi T\).

**A simple interaction check.** If increasing the fee has a different effect at a long maturity than at a short maturity, a model with one fee curve shared across all maturities may be limited. Adding a scientifically motivated interaction feature is one possible extension, tested in homework.

The spline transformer is fitted on reference features inside the model pipeline. It never sees validation responses when creating its basis or estimating coefficients.

## 4. Trees, forests, and an optional boosting model

A regression tree partitions feature space with rules such as “scaled maturity is below a threshold.” Under squared error, a leaf predicts the average training label of the reference contracts assigned to that leaf.

A random forest fits many randomized trees and averages their predictions:

\[
\widehat f_{\mathrm{RF}}(z)=\frac1K\sum_{k=1}^{K}\widehat f_k(z).
\]

Trees can represent interactions through sequences of splits. A small leaf can fit detailed structure but may track label noise; a larger leaf averages more observations but can smooth away real changes. We vary the minimum leaf size in a small declared grid.

Gradient boosting, the optional model, builds a sequence of shallow trees that correct errors left by the current predictor:

\[
f_{t+1}(z)=f_t(z)+\eta h_t(z),
\]

where \(\eta\) is the learning rate. We do not need to implement the optimization to compare the method responsibly.

| Predictor | Main structure | A question to investigate |
|---|---|---|
| CSM-style 1NN | Copies one Euclidean nearest reference | Does abrupt copying cause large local errors? |
| Uniform 3NN | Averages three nearest references | Does smoothing help or blur relevant differences? |
| Ridge | One linear response | Is simplicity adequate for a small label bank? |
| SplineGAM | Sum of nonlinear feature contributions | Are missing interactions limiting accuracy? |
| Random forest | Average of partition-based predictors | Is the reference sample large enough? |
| Gradient boosting, optional | Sequential shallow-tree corrections | Does extra flexibility justify its cost? |

No row is a predeclared winner. The comparison is about observed accuracy and work under a specified data design.

## 5. A controlled validation experiment

The same reference indices, MC means, feature transformation, and validation MC means are reused by every model. Only the predictor and its stated tuning parameters change.

For each family, fit each parameter setting using reference labels only, predict validation contracts, and choose the smallest validation RMSE:

\[
\widehat h
=\arg\min_{h\in\mathcal H}
\sqrt{\frac1v\sum_{q=1}^{v}
\left(f_h(z_q)-\widehat y^{\,\mathrm{MC}}_{\mathrm{val},q}\right)^2}.
\]

Validation labels are allowed to select settings. They are not appended to the reference training set in this exercise. The test set is neither scored nor inspected.

The core grids contain three settings each: Ridge strength, spline Ridge strength, and forest minimum leaf size. The 1NN and 3NN baselines have fixed settings. We charge each family's full search work, not only the time of its best setting. The families receive the same simulation label budget; their computation costs may differ.

**Avoid two common mistakes.** Low training error can reflect fitting noise, and the best validation score has benefited from selection. Neither is the final generalization result. Weeks 7–8 separate development and a frozen test assessment.

Raw predictions are scored consistently. We report negative or above-bound predictions separately. If you later add clipping to the model, declare and evaluate that rule for every affected method before opening the test set.

In [ ]:
base_models = make_regressors(seed=2027)
parameter_grids = {
    "Ridge": [{"alpha": a} for a in [0.1, 1.0, 10.0]],
    "SplineGAM": [{"ridge__alpha": a} for a in [0.1, 1.0, 10.0]],
    "RandomForest": [{"min_samples_leaf": leaf} for leaf in [1, 3, 5]],
}
if INCLUDE_OPTIONAL_BOOSTING:
    parameter_grids["GradientBoosting"] = [
        {"max_depth": depth} for depth in [1, 2, 3]
    ]

all_trials = []
selected_models = {}
family_search_seconds = {}

for family, grid in parameter_grids.items():
    search_start = perf_counter()
    local_trials = []
    for parameters in grid:
        estimator = clone(base_models[family]).set_params(**parameters)

        t0 = perf_counter()
        estimator.fit(Zreference, y_reference)
        fit_seconds = perf_counter() - t0

        t0 = perf_counter()
        prediction = estimator.predict(Zvalidation)
        prediction_seconds = perf_counter() - t0

        train_prediction = estimator.predict(Zreference)
        validation_metrics = score_predictions(
            y_validation, prediction, premium=premium_validation
        )
        train_metrics = score_predictions(y_reference, train_prediction)
        row = {
            "family": family,
            "parameters": json.dumps(parameters, sort_keys=True),
            "train_RMSE_pp": 100 * train_metrics["rmse"],
            "validation_RMSE_pp": 100 * validation_metrics["rmse"],
            "fit_seconds": fit_seconds,
            "prediction_seconds": prediction_seconds,
        }
        all_trials.append(row)
        local_trials.append({
            "row": row, "estimator": estimator, "prediction": prediction,
            "metrics": validation_metrics
        })

    selected_models[family] = min(
        local_trials,
        key=lambda entry: (
            entry["metrics"]["rmse"], entry["row"]["parameters"]
        )
    )
    family_search_seconds[family] = perf_counter() - search_start

trial_table = pd.DataFrame(all_trials)
trial_table["selected_within_family"] = [
    row["parameters"] == selected_models[row["family"]]["row"]["parameters"]
    for row in all_trials
]
display(trial_table.round(6))

In [ ]:
predictions = {}
leaderboard_rows = []
shared_development_seconds = (
    feature_seconds + compression_seconds
    + reference_mc_seconds + validation_mc_seconds
)

# The nearest-reference baselines use the identical label bank.
for name, k in [("CSM1NN", 1), ("kNN3", 3)]:
    method_start = perf_counter()
    t0 = perf_counter()
    pred, neighbor_ids = nearest_predict(
        Zreference, y_reference, Zvalidation, k=k
    )
    prediction_seconds = perf_counter() - t0
    metric = score_predictions(y_validation, pred, premium=premium_validation)
    method_seconds = perf_counter() - method_start
    predictions[name] = pred
    leaderboard_rows.append({
        "predictor": name, "selected_parameters": f"k={k}",
        "validation_RMSE_pp": 100 * metric["rmse"],
        "validation_MAE_pp": 100 * metric["mae"],
        "validation_p95_abs_pp": 100 * metric["p95_abs"],
        "validation_bias_pp": 100 * metric["bias"],
        "validation_aggregate_error_pct": 100 * metric["portfolio_relative_error"],
        "negative_count": int(np.sum(pred < 0)),
        "above_contract_bound_count": int(np.sum(pred > upper_bound_validation)),
        "tuning_candidates": 1,
        "chosen_fit_seconds": 0.0,
        "chosen_prediction_seconds": prediction_seconds,
        "family_search_seconds": method_seconds,
        "standalone_development_seconds": shared_development_seconds + method_seconds,
    })

for family, selected in selected_models.items():
    pred = selected["prediction"]
    metric = selected["metrics"]
    predictions[family] = pred
    leaderboard_rows.append({
        "predictor": family,
        "selected_parameters": selected["row"]["parameters"],
        "validation_RMSE_pp": 100 * metric["rmse"],
        "validation_MAE_pp": 100 * metric["mae"],
        "validation_p95_abs_pp": 100 * metric["p95_abs"],
        "validation_bias_pp": 100 * metric["bias"],
        "validation_aggregate_error_pct": 100 * metric["portfolio_relative_error"],
        "negative_count": int(np.sum(pred < 0)),
        "above_contract_bound_count": int(np.sum(pred > upper_bound_validation)),
        "tuning_candidates": len(parameter_grids[family]),
        "chosen_fit_seconds": selected["row"]["fit_seconds"],
        "chosen_prediction_seconds": selected["row"]["prediction_seconds"],
        "family_search_seconds": family_search_seconds[family],
        "standalone_development_seconds": (
            shared_development_seconds + family_search_seconds[family]
        ),
    })

leaderboard = pd.DataFrame(leaderboard_rows).sort_values(
    "validation_RMSE_pp", kind="stable"
).reset_index(drop=True)

display(leaderboard[[
    "predictor", "selected_parameters", "validation_RMSE_pp",
    "validation_MAE_pp", "validation_p95_abs_pp", "validation_bias_pp",
    "validation_aggregate_error_pct", "negative_count",
    "above_contract_bound_count"
]].round(5))
print("This is a validation development table, not a final test ranking.")

### Why the bound check matters

Because each discounted top-up payoff lies between zero and \(g_qe^{-rT_q}\),

\[
0\leq y(q)\leq g_qe^{-rT_q}.
\]

These bounds come from the payoff itself and require no exact-price label. Violations identify an economically inconsistent raw prediction, even if the average squared error is small. Different contracts have different upper bounds, so even copying a valid reference response can exceed the query's bound.

The code reports all raw models using one scoring convention. A predeclared projection onto the query's admissible interval is a sensible later extension. It is a modelling rule that must be documented, not silently applied to a favored method.

In [ ]:
plot_names = ["CSM1NN", "Ridge", "SplineGAM", "RandomForest"]
fig, axes = plt.subplots(2, 2, figsize=(10, 8.2))
for ax, name in zip(axes.ravel(), plot_names):
    pred = predictions[name]
    all_values = 100 * np.concatenate([y_validation, pred])
    lo, hi = float(all_values.min()), float(all_values.max())
    ax.scatter(100 * y_validation, 100 * pred,
               s=23, alpha=0.7, color="#356f8f")
    ax.plot([lo, hi], [lo, hi], "--", color="#777777", linewidth=1)
    ax.set(title=name, xlabel="Validation MC mean (% of premium)",
           ylabel="Predicted liability (% of premium)")
fig.suptitle("Same references and labels; different prediction rules", y=1.01)
fig.tight_layout()
plt.show()

# Model size is observable; it is not a performance conclusion.
spline_basis = selected_models["SplineGAM"]["estimator"].named_steps[
    "splinetransformer"
].transform(Zreference)
print("Reference rows:", Zreference.shape[0])
print("Original features:", Zreference.shape[1])
print("Additive spline basis columns:", spline_basis.shape[1])

ridge_estimator = selected_models["Ridge"]["estimator"]
display(pd.DataFrame({
    "feature": FEATURES,
    "selected_Ridge_coefficient": ridge_estimator.coef_
}))

## 6. Interpret the comparison and keep the accounting complete

A useful result explains a pattern rather than only naming the top row.

- If a flexible model has much lower training error but little validation improvement, ask whether its sample size and label noise limit generalization.
- If an additive model improves on a linear model, inspect whether nonlinear feature responses may matter; this alone does not identify the correct mechanism.
- If individual errors remain large while aggregate error is small, cancellation may be masking local failures.
- If a method is slightly more accurate but costs substantially more, decide what accuracy and computing budget the application requires.

These are prompts for your observed outputs, not claims about which pattern will appear.

The exact normalized target is unknown to the fitting workflow. Validation MC uncertainty contributes to every score. A small difference between methods may be unstable across simulation or compressor seeds. Repeated validation exploration is legitimate development, but it strengthens the need for the sealed test assessment.

The table below separates shared label generation from model fitting and search. These exploratory comparisons hold paths per reference fixed; the final benchmark will also charge validation against a declared total development path budget.

In [ ]:
label_ledger = pd.DataFrame([
    {
        "label_bank": "Reference (shared by all predictors)",
        "contract_valuations": len(references),
        "paths_per_contract": REFERENCE_PATHS,
        "payoff_paths": len(references) * REFERENCE_PATHS,
        "simulation_seconds": reference_mc_seconds
    },
    {
        "label_bank": "Validation (shared by all tuning trials)",
        "contract_valuations": len(validation),
        "paths_per_contract": VALIDATION_PATHS,
        "payoff_paths": len(validation) * VALIDATION_PATHS,
        "simulation_seconds": validation_mc_seconds
    }
])
display(label_ledger.round(5))
display(leaderboard[[
    "predictor", "tuning_candidates", "chosen_fit_seconds",
    "chosen_prediction_seconds", "family_search_seconds",
    "standalone_development_seconds"
]].round(6))
print("Reference-selection seconds:", round(compression_seconds, 6))
print("Feature-preparation seconds:", round(feature_seconds, 6))
print("Actual guided label-generation payoff paths:",
      int(label_ledger["payoff_paths"].sum()))

# Check a potential bias pattern on validation data only.
bias_rows = []
maturity_band = pd.qcut(validation["maturity"], q=3, duplicates="drop")
for name, pred in predictions.items():
    diagnostic = pd.DataFrame({
        "maturity_band": maturity_band.astype(str).to_numpy(),
        "error_pp": 100 * (pred - y_validation)
    })
    for band, group in diagnostic.groupby("maturity_band"):
        bias_rows.append({
            "predictor": name, "maturity_band": band, "contracts": len(group),
            "bias_pp": group["error_pp"].mean(),
            "MAE_pp": group["error_pp"].abs().mean()
        })
display(pd.DataFrame(bias_rows).round(5))

**Accounting interpretation.** A standalone development time charges that model family's full search plus the common preprocessing, compressor, reference MC, and validation MC. The chosen fit is already part of the search; it is not charged a second time. If these fitted models are reused, subsequent prediction cost is smaller than starting the workflow again.

When reporting the actual cost of this multi-model campaign, count common label generation once. Do not sum standalone end-to-end rows, which intentionally repeat shared costs for comparable individual workflows. Record the number of parameter candidates as well as wall-clock time.

**Write a 200-word development note.** State the common data design; name two quantitative observations; discuss one approximation limitation and one uncertainty limitation; specify which models and settings should be considered for the final protocol. Do not call validation RMSE “test accuracy.”

## Reading assignment — approximately 45–60 minutes

1. **Ben Mingbin Feng, Zhenni Tan, and Jiayi Zheng (2020), _Efficient Simulation Designs for Valuation of Large Variable Annuity Portfolios_.** [Author PDF](https://www.math.uwaterloo.ca/~mbfeng/papers/2020_VAPort.pdf); [DOI](https://doi.org/10.1080/10920277.2019.1685394). Revisit Section 5, focusing on what a predictor receives and returns.
2. **scikit-learn official documentation:** [Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html), [SplineTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.SplineTransformer.html), and [RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html). Read the short descriptions and the parameters used in this notebook; exhaustive API reading is unnecessary.

**Focused questions**

- What information is shared across the CSM benchmark and a fitted regression predictor?
- Which parameter controls each model's flexibility here? What risk arises when only 40 reference labels are available?
- Why is an additive spline model nonlinear without modelling every interaction?
- Why do model selection and final assessment need separate contract sets?
- Give one reason a raw prediction can violate a payoff bound, even when every training label satisfies its own contract's bound.

The contract-similarity research manuscript is not part of the required student reading.

## Homework 1 — Trace the regularization trade-off

**Core; about 45–60 minutes.** Extend the Ridge strengths below to a logarithmic range from 0.001 to 100. Plot training and validation RMSE against the strength on a logarithmic horizontal axis. Describe the pattern you actually see, including the possibility that it is weak or non-monotone.

Reuse the same labels and record how many settings you tried. Treat the additional search as development work; do not touch the test data.

In [ ]:
homework_alphas = [0.03, 0.3, 3.0]  # Extend this declared grid for homework.
regularization_rows = []
for alpha in homework_alphas:
    model = Ridge(alpha=alpha).fit(Zreference, y_reference)
    train_prediction = model.predict(Zreference)
    validation_prediction = model.predict(Zvalidation)
    regularization_rows.append({
        "alpha": alpha,
        "train_RMSE_pp": 100 * score_predictions(
            y_reference, train_prediction
        )["rmse"],
        "validation_RMSE_pp": 100 * score_predictions(
            y_validation, validation_prediction
        )["rmse"],
        "coefficient_L2_norm": float(np.linalg.norm(model.coef_))
    })
display(pd.DataFrame(regularization_rows).round(6))

## Homework 2 — Add one justified interaction feature

**Core; about 60 minutes.** The fee enters the account model through \(\phi T\). The starter adds fee-times-maturity as a sixth feature and fits the same additive spline family.

Keep the reference indices and MC labels unchanged. Fit the new feature scaler on candidates only. Compare the original and augmented model using the same declared penalty setting, then decide whether a small, explicitly charged validation search is justified.

Explain why this feature contains contract information rather than a hidden valuation label. As a second investigation, replace it with \(\sigma\sqrt T\) and document the change. Do not add many features merely because repeated validation experiments eventually find a favorable score.

In [ ]:
augmented_candidate = candidate[FEATURES].copy()
augmented_validation = validation[FEATURES].copy()
for frame in [augmented_candidate, augmented_validation]:
    frame["fee_times_maturity"] = frame["fee"] * frame["maturity"]

augmented_scaler = MinMaxScaler().fit(augmented_candidate)
Zaugmented_candidate = augmented_scaler.transform(augmented_candidate)
Zaugmented_validation = augmented_scaler.transform(augmented_validation)
Zaugmented_reference = Zaugmented_candidate[reference_positions]

fixed_alpha = 1.0  # Same predeclared penalty for both variants in this comparison.
original_model = clone(base_models["SplineGAM"]).set_params(
    ridge__alpha=fixed_alpha
).fit(Zreference, y_reference)
augmented_model = clone(base_models["SplineGAM"]).set_params(
    ridge__alpha=fixed_alpha
).fit(Zaugmented_reference, y_reference)

feature_extension_rows = []
for label, estimator, Zq in [
    ("Original five features", original_model, Zvalidation),
    ("Five features plus fee*T", augmented_model, Zaugmented_validation)
]:
    pred = estimator.predict(Zq)
    feature_extension_rows.append({
        "feature_design": label, "ridge_alpha": fixed_alpha,
        "validation_RMSE_pp": 100 * score_predictions(
            y_validation, pred
        )["rmse"]
    })
display(pd.DataFrame(feature_extension_rows).round(6))

## Homework 3 — A boosting challenger (optional)

Fit the provided small gradient boosting model on the same reference bank. Compare depths 1, 2, and 3, charging all fitting time. Explain what additional question this model addresses and whether its observed validation result warrants retaining it for the final benchmark.

The starter fits one declared setting. Increasing the number of models does not create additional independent evidence.

In [ ]:
t0 = perf_counter()
boosting_starter = clone(base_models["GradientBoosting"]).fit(
    Zreference, y_reference
)
boosting_fit_seconds = perf_counter() - t0
boosting_prediction = boosting_starter.predict(Zvalidation)
display(pd.DataFrame([{
    "model": "GradientBoosting",
    "max_depth": boosting_starter.get_params()["max_depth"],
    "fit_seconds": boosting_fit_seconds,
    "validation_RMSE_pp": 100 * score_predictions(
        y_validation, boosting_prediction
    )["rmse"]
}]).round(6))

## Weekly deliverable checklist

- [ ] I can distinguish a value predictor from a learned distance.
- [ ] All predictors use identical reference indices and identical MC labels.
- [ ] I can explain linear, additive nonlinear, and interaction-capable models.
- [ ] My tuning table shows every attempted core configuration.
- [ ] Validation responses select settings but do not enter model fitting.
- [ ] I distinguish training RMSE, validation RMSE, and the later final test result.
- [ ] I report units, MC uncertainty, and any raw payoff-bound violations.
- [ ] My cost ledger includes all search candidates and shared validation labels.
- [ ] I completed the reading questions and the two core homework exercises.
- [ ] The test set remains closed and no exact-price labels entered fitting or tuning.

**Bring to the meeting:** your development leaderboard, one diagnostic plot, a 200-word interpretation, and a proposed hypothesis about when a learned distance might improve on Euclidean neighbor copying.